> **Notebook-first lesson.** Run cells in order. The final activity is designed to be changed and rerun.

## Mathematical Framework

Math companions for this lesson:

- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

Track **shapes, assumptions, objective, derivatives/updates, and the conditions under which the derivation stops matching reality**.

# Lesson 28: GPU training and reproducibility

## Device-aware code


In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(SEED)
X_train = torch.randn(160, 2)
y_train = (X_train[:, 0] + .5 * X_train[:, 1] > 0).long()
X_val = torch.randn(60, 2)
y_val = (X_val[:, 0] + .5 * X_val[:, 1] > 0).long()
device = torch.device('cpu')
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.01)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

for xb, yb in loader:
    xb = xb.to(device)
    yb = yb.to(device)

    logits = model(xb)



## GPU mental model
A GPU is valuable when the workload contains large amounts of parallel numerical work. Moving tiny arrays back and forth can cost more than the computation itself.

## Reproducibility


In [ ]:
import random
import numpy as np
import torch

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)



Exact determinism can depend on hardware, kernels and library settings.

## Measure, do not assume
Time:
- data loading;
- host-to-device transfer;
- forward pass;
- backward pass;
- optimizer step.

## Exercise
Benchmark the same MLP on CPU and GPU at increasing batch sizes. Explain where GPU acceleration starts becoming useful.


## Runnable activity
Run this experiment. Then change one architectural, data, or optimization choice and compare.

In [ ]:
import random, numpy as np, torch, time
seed=42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:",device)
for n in [128,512,2048]:
    a=torch.randn(n,n,device=device); b=torch.randn(n,n,device=device)
    if device.type=="cuda": torch.cuda.synchronize()
    t=time.perf_counter(); c=a@b
    if device.type=="cuda": torch.cuda.synchronize()
    print("n",n,"seconds",time.perf_counter()-t)

## Explanation checkpoint
Add a Markdown cell that explains the tensor shapes, the mechanism being tested, and what changed when you modified the experiment.